## Extend CDR Sequence Files with Calculated Features
---

##  1. Create properties of the AS in Dictionary

In [1]:
import pandas as pd

amino_acid_properties = {
    'A': {'hydrophobicity': 1.8, 'charge': 0, 'mass': 71.08, 'polarity': 8.1 }, #Alanin
    'R': {'hydrophobicity': -4.5, 'charge': +1, 'mass': 156.19, 'polarity': 10.5},# Arginin
    'N': {'hydrophobicity': -3.5, 'charge': 0, 'mass': 114.11, 'polarity': 11.6},#Asparagin
    'D': {'hydrophobicity': -3.5, 'charge': -1, 'mass': 115.09, 'polarity': 13.0},#Asparaginsäure
    'C': {'hydrophobicity': 2.5, 'charge': 0, 'mass': 103.15, 'polarity': 5.5},#Cystein
    'Q': {'hydrophobicity': -3.5, 'charge': 0, 'mass': 128.13, 'polarity': 10.5},#Glutamin
    'E': {'hydrophobicity': -3.5, 'charge': -1, 'mass': 129.12, 'polarity': 12.3},#Glutaminsäure
    'G': {'hydrophobicity': -0.4, 'charge': 0, 'mass': 75.05, 'polarity': 9.0},#Glycin
    'H': {'hydrophobicity': -3.2, 'charge': +1, 'mass': 137.14, 'polarity': 10.4},#Histidin
    'I': {'hydrophobicity': 4.5, 'charge': 0, 'mass': 113.16, 'polarity': 5.2},#Isoleucin
    'L': {'hydrophobicity': 3.8, 'charge': 0, 'mass': 113.16, 'polarity': 4.9},#Leucin
    'K': {'hydrophobicity': -3.9, 'charge': +1, 'mass': 128.18, 'polarity': 11.3},#Lysin
    'M': {'hydrophobicity': 1.9, 'charge': 0, 'mass': 131.20, 'polarity': 5.7},#Methionin
    'F': {'hydrophobicity': 2.8, 'charge': 0, 'mass': 147.18, 'polarity': 5.2},#Phenylalanin
    'P': {'hydrophobicity': -1.6, 'charge': 0, 'mass': 97.12, 'polarity': 8.0},#Prolin
    'S': {'hydrophobicity': -0.8, 'charge': 0, 'mass': 87.08, 'polarity': 9.2},#Serin
    'T': {'hydrophobicity': -0.7, 'charge': 0, 'mass': 101.11, 'polarity': 8.6},#Threonim
    'W': {'hydrophobicity': -0.9, 'charge': 0, 'mass': 186.22, 'polarity': 5.4},#Tryptophan
    'Y': {'hydrophobicity': -1.3, 'charge': 0, 'mass': 163.18, 'polarity': 6.2},#Tyrosin
    'V': {'hydrophobicity': 4.2, 'charge': 0, 'mass': 99.13, 'polarity': 5.9}#Valin
}

#### woher stammen die Werte:

Hydrophobicity: 
- Kyte J, Doolittle RF. A simple method for displaying the hydropathic character of a protein. J Mol Biol. 1982 May 5;157(1):105-32
---

charge:
- Die Werte basieren auf dem Ladungszustand der Seitenkette bei neutralem pH (~7.0)
- Zuordnung ist biochemisch fest definiert:
=> +1: positiv geladen (Arg, Lys, His)
=> -1: negativ geladen (Asp, Glu)
=> 0: ungeladen (alle anderen)
---

mass: [g/mol]
- https://www.sigmaaldrich.com/DE/de/technical-documents/technical-article/protein-biology/protein-structural-analysis/amino-acid-reference-chart?srsltid=AfmBOoqS9jHqRJYPEYQ19QbSpXnpfRSnIhBC5KOjPBVqHMXzcIJfQvVl
- mittleren molaren Masse der freien Aminosäure aus Tabelle entnommen (Rückstandsgewicht - H_20)
---

polarity:
- Grantham, R. (1974). Amino acid difference formula to help explain protein evolution.
Science, 185(4154), 862–864.
---

#### Überblick der Eigenschaften
- besonders hydrophob: Isoleucin, Valin, Leucin, Phenylalanin, Cystein, Methionin, Alanin ...
- besonders Hydrophil: Arginin, Lysin, Asparagin, Asparaginsäure, Histidin ...
- pos. geladen (basisch): Arginin, Histidin, Lysin
- neg. geladen (sauer): Asparaginsäure, Glutaminsäure
- besonders schwere AS: Tryptophan, Tyrosin, Arginin, Phenylalanin, Histidin ..
- besonders leichte AS: Alanin, Glycin, Serin, Prolin ...
- besonders polar: Asparaginsäure, Glutaminsäure, Asparagin, Lysin, Glutamin, Histidin ...
- besonders unpolare: Leucin, Phenylalanin, Tryptophan, Cystein ...
---

## 2. Count amino acid frequencies in CDR regions and save them 

In [8]:
# AS Liste erzeugen um später über alle AS itrrieren zu können
amino_acids = list(amino_acid_properties.keys())  # Liste der Aminosäure-Kurzzeichen aus dem Dictionary wird erstellt, .keysholt AS-Abkürzungen 

# Liste mit Pfaden zu den CDR-Sequenzdateien für verschiedene Gruppen
files = [
    "../generated/cdrs/seq/human_cdr_seq.tsv",       # Datei mit humanen CDR-Sequenzen
    "../generated/cdrs/seq/influenza_cdr_seq.tsv",   # Datei mit Influenza-Sequenzen
    "../generated/cdrs/seq/corona_cdr_seq.tsv"       # Datei mit SARS-CoV-2-Sequenzen
]

# Schleife über alle Dateien in der Liste
for file in files:  # für jede Datei in "files"
    df = pd.read_csv(file, sep="\t")  # jeweilige Datei einlesen als DataFrame (Tabulator-getrennt)

    # Überprüfung und Verarbeitung jeder CDR-Region
    for cdr in ['CDR_H1', 'CDR_H2', 'CDR_H3']:  # über CDR_H1, CDR_H2 und CDR_H3 iterieren in df
        if cdr in df.columns:  # prüfen, ob die CDR-Spalte in der Datei vorhanden ist
           

            # Für jede Sequenz: Zähle alle Aminosäuren
            for index, row in df.iterrows():  # Schleife: Zeilenweise durch das DataFrame iterieren
                seq = row[cdr]  # aktuelle Sequenz aus der CDR-Spalte
                for aa in amino_acids:  # Schleife prüft jede Aminosäure einzeln
                    count = seq.count(aa)  # zähle, wie oft die Aminosäure in der Sequenz vorkommt; count speichert anzahl
                    # df.loc[...] = schreibt Wert an bestimmte Stelle in der Tabell
                    # index	= Zeilennumme
                    # f'{cdr}_count_{aa}'	= neue Spalte mit Namen 
                    # = count	= trägt die gezählte Zahl ein
                    df.loc[index, f'{cdr}_count_{aa}'] = count  # neue Spalte: z. B. "CDR_H1_count_A"

        else:
            print(f"   Spalte {cdr} fehlt in Datei: {file}")  # Meldung, falls die Spalte nicht existiert

    # Überschreibe die Datei direkt mit den neuen Daten
    # df.to_csv(...) = 	speichert die bearbeitete Tabelle wieder ab
    # file	= ist der ursprüngliche Dateipfad
    # sep="\t"	= verwendet Tabulator als Trennzeichen (für .tsv-Dateien)
    # index=False	= speichert nicht die Pandas-Zeilennummern
    df.to_csv(file, sep="\t", index=False)  # speichert die Datei ohne Indexspalte
    print(f"   File updated and saved: {file}")  # Bestätigung der Speicherung


   File updated and saved: ../generated/cdrs/seq/human_cdr_seq.tsv
   File updated and saved: ../generated/cdrs/seq/influenza_cdr_seq.tsv
   File updated and saved: ../generated/cdrs/seq/corona_cdr_seq.tsv


## 3. Calculating the average amino acid properties in CDR sequences ( hydrophobicity, charge,polarity, mass)

In [9]:
for file in files:  # Schleife über jede Datei in der Liste "files"
    print(f"\nCalculate average properties for: {file}")  # Gibt aus, welche Datei gerade verarbeitet wird
    df = pd.read_csv(file, sep="\t")  # Lese die TSV-Datei als DataFrame ein (Tabulator-getrennt)

    for cdr in ['CDR_H1', 'CDR_H2', 'CDR_H3']:  # Schleife über die drei CDR-Regionen
        if cdr in df.columns:  # Prüfe, ob die aktuelle CDR-Spalte in der Tabelle existiert
            for index, row in df.iterrows():  # Iteriere über jede Zeile der Tabelle
                sequence = row[cdr]  # Extrahiere die Sequenz aus der aktuellen Zeile in der Spalte "CDR_H1" (oder H2, H3)

                # leere listen die später gefüllt werden
                hydrophobicities = []  # Liste zum Zwischenspeichern der Hydrophobizitätswerte
                charges = []  # Liste zum Zwischenspeichern der Ladungen
                polarities = []  # Liste zum Zwischenspeichern der Polaritätswerte
                masses = []  # Liste zum Zwischenspeichern der Massen

                # Schleife über jedes einzelne Zeichen der Sequenz
                for aa in sequence:  # Gehe jedes Aminosäure-Zeichen in der Sequenz durch
                    if aa in amino_acid_properties:  # Prüfe, ob die Aminosäure im Eigenschaften-Dictionary enthalten ist

                        # Werte der AS werden in Listen geschrieben
                        hydrophobicities.append(amino_acid_properties[aa]['hydrophobicity'])  # Hydrophobizität hinzufügen
                        charges.append(amino_acid_properties[aa]['charge'])  # Ladung hinzufügen
                        polarities.append(amino_acid_properties[aa]['polarity'])  # Polarität hinzufügen
                        masses.append(amino_acid_properties[aa]['mass'])  # Masse hinzufügen
                    else:
                        print(f" Unknown amino acid '{aa}' in {cdr}, row {index+1}")  # Warnung bei unbekannter Aminosäure

                # Berechnung und Abspeichern der Mittelwerte (nur wenn Liste nicht leer ist)
                if hydrophobicities:
                    df.loc[index, f'{cdr}_mean_hydrophobicity'] = sum(hydrophobicities) / len(hydrophobicities)
                if charges:
                    df.loc[index, f'{cdr}_mean_charge'] = sum(charges) / len(charges)
                if polarities:
                    df.loc[index, f'{cdr}_mean_polarity'] = sum(polarities) / len(polarities)
                if masses:
                    df.loc[index, f'{cdr}_mean_mass'] = sum(masses) / len(masses)
                # df.loc[index, 'spalte']	= Zugriff/Zuweisung zu Zelle
                # index	= Aktuelle Zeile der Schleife
                # f'{cdr}_mean_mass'	= Neue Spalte pro Region (z. B. CDR_H3_mean_mass)
                # sum(...) / len(...)	= Durchschnittswert berechnen

    # Speichere die aktualisierte Datei (mit neuen Mittelwert-Spalten) ohne Index-Spalte
    df.to_csv(file, sep="\t", index=False)
    print(f"Properties directly in {file} saved")  # Bestätigung: Datei erfolgreich erweitert und gespeichert



Calculate average properties for: ../generated/cdrs/seq/human_cdr_seq.tsv
Properties directly in ../generated/cdrs/seq/human_cdr_seq.tsv saved

Calculate average properties for: ../generated/cdrs/seq/influenza_cdr_seq.tsv
Properties directly in ../generated/cdrs/seq/influenza_cdr_seq.tsv saved

Calculate average properties for: ../generated/cdrs/seq/corona_cdr_seq.tsv
Properties directly in ../generated/cdrs/seq/corona_cdr_seq.tsv saved


## 4. Calculate and Add CDR Lengths to Existing Sequence Files

In [10]:
# CDR-Längen berechnen und direkt in bestehenden Dateien speichern
cdrs = ["CDR_H1", "CDR_H2", "CDR_H3"]  # Liste der zu analysierenden CDR-Regionen

for file in files:  # Schleife über jede Datei in der Liste "files"
    print(f"\nCalculate CDR lengths for: {file}")  # Gibt aus, welche Datei gerade verarbeitet wird
    df = pd.read_csv(file, sep="\t")  # Lese die TSV-Datei mit Pandas ein (tab-getrennte Datei)

    for cdr in cdrs:  # Schleife über jede CDR-Region (CDR_H1, CDR_H2, CDR_H3)
        print(f" -> Processed: {cdr}")  # Terminalausgabe: CDR wird verarbeitet
        df[f'{cdr}_length'] = df[cdr].apply(len)  # Neue Spalte: Länge der Sequenz mit .apply(len)
        # Länge der Aminosäuresequenz wird berechnet

    df.to_csv(file, sep="\t", index=False)  # Überschreibe die Datei mit den neuen Spalten (ohne Index)
    print(f"Saved updated: {file}")  # Terminalausgabe: Datei wurde erfolgreich gespeichert



Calculate CDR lengths for: ../generated/cdrs/seq/human_cdr_seq.tsv
 -> Processed: CDR_H1
 -> Processed: CDR_H2
 -> Processed: CDR_H3
Saved updated: ../generated/cdrs/seq/human_cdr_seq.tsv

Calculate CDR lengths for: ../generated/cdrs/seq/influenza_cdr_seq.tsv
 -> Processed: CDR_H1
 -> Processed: CDR_H2
 -> Processed: CDR_H3
Saved updated: ../generated/cdrs/seq/influenza_cdr_seq.tsv

Calculate CDR lengths for: ../generated/cdrs/seq/corona_cdr_seq.tsv
 -> Processed: CDR_H1
 -> Processed: CDR_H2
 -> Processed: CDR_H3
Saved updated: ../generated/cdrs/seq/corona_cdr_seq.tsv


### control

In [5]:
# Load data set (e.g. SARS-CoV-2)
df = pd.read_csv( "../generated/cdrs/seq/corona_cdr_seq.tsv", sep="\t")

# Show only the first 5 lines
df.head(5)

,pdb,heavy_chain,CDR_H3,CDR_H2,CDR_H1,CDR_H1_count_A,CDR_H1_count_R,CDR_H1_count_N,CDR_H1_count_D,CDR_H1_count_C,...,CDR_H3_mean_hydrophobicity,CDR_H3_mean_charge,CDR_H3_mean_polarity,CDR_H3_mean_mass,cdr_h1_length,cdr_h2_length,cdr_h3_length,CDR_H1_length,CDR_H2_length,CDR_H3_length
0,9cci,B,TFGTYYDNTEDWFFDF,YGGDSD,GYSFSSF,0.0,0.0,0.0,0.0,0.0,...,-0.768750,-0.250000,8.518750,129.261250,7,6,16,7,6,16
1,9ccj,H,LPLGERIDY,YYSGT,GGSINTNMY,0.0,0.0,2.0,0.0,0.0,...,-0.300000,-0.111111,8.222222,119.470000,9,5,9,9,5,9
2,9bj2,H,HNGDPYDFWSGYNTWAGGLDV,IPFLDV,GVIFSRN,0.0,1.0,1.0,0.0,0.0,...,-0.819048,-0.095238,8.652381,115.499524,7,6,21,7,6,21
3,9bj3,C,VPQAGAAQGHYYYYYGMDV,IPILGI,GGTFINY,0.0,0.0,1.0,0.0,0.0,...,-0.384211,0.000000,8.010526,115.229474,7,6,19,7,6,19
4,8z6r,E,QGDLGDWILLGY,YPGDSD,GYTFSYY,0.0,0.0,0.0,0.0,0.0,...,0.166667,-0.166667,7.916667,115.458333,7,6,12,7,6,12


### see which columns we have now

In [6]:
# Show all column names
print(df.columns.tolist())


['pdb', 'heavy_chain', 'CDR_H3', 'CDR_H2', 'CDR_H1', 'CDR_H1_count_A', 'CDR_H1_count_R', 'CDR_H1_count_N', 'CDR_H1_count_D', 'CDR_H1_count_C', 'CDR_H1_count_Q', 'CDR_H1_count_E', 'CDR_H1_count_G', 'CDR_H1_count_H', 'CDR_H1_count_I', 'CDR_H1_count_L', 'CDR_H1_count_K', 'CDR_H1_count_M', 'CDR_H1_count_F', 'CDR_H1_count_P', 'CDR_H1_count_S', 'CDR_H1_count_T', 'CDR_H1_count_W', 'CDR_H1_count_Y', 'CDR_H1_count_V', 'CDR_H2_count_A', 'CDR_H2_count_R', 'CDR_H2_count_N', 'CDR_H2_count_D', 'CDR_H2_count_C', 'CDR_H2_count_Q', 'CDR_H2_count_E', 'CDR_H2_count_G', 'CDR_H2_count_H', 'CDR_H2_count_I', 'CDR_H2_count_L', 'CDR_H2_count_K', 'CDR_H2_count_M', 'CDR_H2_count_F', 'CDR_H2_count_P', 'CDR_H2_count_S', 'CDR_H2_count_T', 'CDR_H2_count_W', 'CDR_H2_count_Y', 'CDR_H2_count_V', 'CDR_H3_count_A', 'CDR_H3_count_R', 'CDR_H3_count_N', 'CDR_H3_count_D', 'CDR_H3_count_C', 'CDR_H3_count_Q', 'CDR_H3_count_E', 'CDR_H3_count_G', 'CDR_H3_count_H', 'CDR_H3_count_I', 'CDR_H3_count_L', 'CDR_H3_count_K', 'CDR_H3_cou